# RSS LD Sketch Pipeline

## Overview
This pipeline generates a **stochastic genotype sample** U from whole-genome sequencing VCF files and stores it as a PLINK2 pgen file for use as an LD reference panel.

**Terminology:**
- **Stochastic genotype sample**: U = W<sup>T</sup> G, shape (B × p). B pseudo-individuals × p variants. This is what we store in the pgen file (after scaling to $[0,2]$).
- **LD sketch** (computed downstream, never stored here): R̂ = U<sup>T</sup>U / B, computed by PLINK2 or SuSiE-RSS when LD is requested from the pgen.

**Steps:**
1. Generate projection matrix W once from total cohort sample size n (saved as `.npy`, shared across all chromosomes)
2. Read LD block BED file and write one coordinate file per block
3. For each block in parallel: load VCF variants in `[block_start, block_end)`, compute U = W<sup>T</sup> G, scale to $[0,2]$, stream to PLINK2 pgen format, save allele frequencies
**Matrix convention:**
- G : (n × p) — individuals × variants
- W : (n × B) — individuals × pseudo-individuals, W ~ N(0, 1/√n)
- U : (B × p) — stochastic genotype sample = W<sup>T</sup> G

## Input Files

| File | Description |
|------|-------------|
| VCF (.bgz) | Per-chromosome VCF files with tabix index. If the cohort contains mixed ancestries, provide `--sample-list` to restrict to one source population. |
| `sample_list.txt` | Optional. One sample ID per line. If provided, only these samples are loaded from the VCF. |
| `EUR_LD_blocks.bed` | LD block BED file (0-based half-open, e.g. Berisa & Pickrell EUR blocks). |

## Parameters

**Global parameters:**

| Parameter | Default | Description |
|-----------|---------|-------------|
| `plink2_bin` | `plink2` | Path to plink2 binary, or just `plink2` if on `$PATH` |

**`generate_W` workflow parameters:**

| Parameter | Default | Description |
|-----------|---------|-------------|
| `n_samples` | required | Total number of individuals in the cohort (determines W shape) |
| `output_dir` | required | Output directory for `W_B{B}.npy` |
| `B` | `10000` | Number of pseudo-individuals (sketch dimension) |
| `seed` | `123` | Random seed for W generation |

**`process_block` workflow parameters:**

| Parameter | Default | Description |
|-----------|---------|-------------|
| `vcf_base` | required | Directory containing VCF (.bgz) files, one or more per chromosome |
| `vcf_prefix` | required | Shared filename prefix of all VCF files (everything before `chr{N}:` or `chr{N}.`) |
| `ld_block_file` | required | Path to LD block BED file (e.g. `EUR_LD_blocks.bed`) |
| `chrom` | `0` | Chromosome number (1–22), or 0 for all chromosomes |
| `output_dir` | required | Base output directory |
| `W_matrix` | required | Path to W `.npy` output from `generate_W` |
| `B` | `10000` | Number of pseudo-individuals — must match W |
| `sample_list` | `""` | Optional path to a file of sample IDs (one per line) to subset from the VCF. Leave empty to use all samples. |
| `maf_min` | `0.0005` | Minimum minor allele frequency |
| `mac_min` | `5` | Minimum minor allele count |
| `msng_min` | `0.05` | Maximum missingness rate per variant |

## Output Files

| File | Description |
|------|-------------|
| `W_B{B}.npy` | Projection matrix W, shape (n × B). Output of `generate_W`, input to `process_block`. |
| `chr{N}/{chr{N}}_{start}_{end}/{chr{N}}_{start}_{end}_stocld_panel_sorted.pgen` | Per-block stochastic genotype sample U as PLINK2 binary |
| `chr{N}/{chr{N}}_{start}_{end}/{chr{N}}_{start}_{end}_stocld_panel_sorted.pvar` | Per-block variant information |
| `chr{N}/{chr{N}}_{start}_{end}/{chr{N}}_{start}_{end}_stocld_panel_sorted.psam` | Pseudo-individual IDs S1…S{B} |
| `chr{N}/{chr{N}}_{start}_{end}/{chr{N}}_{start}_{end}_stocld.afreq` | Per-block allele frequencies (OBS_CT = 2 × n) |
| `chr{N}/{chr{N}}_{start}_{end}/{chr{N}}_{start}_{end}_stocld_panel.meta` | Per-block metadata (source_n_samples, B, chrom, block_start, block_end) |

**Timing:** ~2–4 hours per chromosome (parallelized across blocks)

## Minimal Working Example


### Step 0: Generate W once — provide your VCF directory, prefix, and LD block file

In [ ]:
sos run pipeline/rss_ld_sketch.ipynb generate_W \
    --n-samples 2000 \
    --output-dir output/rss_ld_sketch \
    --B 10000 \
    --seed 123

### Step 1: Process blocks (SoS-parallel)


In [ ]:
# Process all blocks for one chromosome
sos run pipeline/rss_ld_sketch.ipynb process_block \
    --vcf-base /path/to/vcfs/ \
    --vcf-prefix your.prefix. \
    --ld-block-file /path/to/EUR_LD_blocks.bed \
    --chrom 22 \
    --output-dir output/rss_ld_sketch \
    --W-matrix output/rss_ld_sketch/W_B10000.npy


In [ ]:
# Process all 22 chromosomes
sos run pipeline/rss_ld_sketch.ipynb process_block \
    --vcf-base /restricted/projectnb/xqtl/R4_QC_NHWonly_rm_monomorphic \
    --vcf-prefix gcad.qc.r4.wgs.36361.GATK.2022.08.15.biallelic.genotypes. \
    --ld-block-file /restricted/projectnb/casa/oaolayin/ROSMAP_NIA_geno/EUR_LD_blocks.bed \
    --chrom 0 \
    --output-dir output/rss_ld_sketch \
    --W-matrix output/rss_ld_sketch/W_B10000.npy


In [ ]:
# --chrom 0 means all 22 chromosomes. SoS reads the BED file and dispatches
# one task per block automatically. Use --chrom 1..22 to restrict to one chromosome.


#### Parallelism is controlled by SoS `-j` and task configuration


In [ ]:
# Control concurrency with -j (local) or task queue (cluster):
#
# Local, 8 parallel blocks:
#   sos run ... process_block --chrom 22 -j 8
#
# LSF cluster (recommended for full genome):
#   sos run ... process_block --chrom 0 \
#       -c cluster.yml -q lsf


## Command Interface

In [ ]:
sos run pipeline/rss_ld_sketch.ipynb -h

```
usage: sos run pipeline/rss_ld_sketch.ipynb
               [workflow_name | -t targets] [options] [workflow_options]
  workflow_name:        Single or combined workflows defined in this script
  targets:              One or more targets to generate
  options:              Single-hyphen sos parameters (see "sos run -h" for details)
  workflow_options:     Double-hyphen workflow-specific parameters

Workflows:
  generate_W
  process_block

Global Workflow Options:
  --cwd output (as path)
  --job-size 1 (as int)
  --walltime 4h
  --mem 32G
  --numThreads 8 (as int)
  --container ''
  --plink2-bin plink2
                       

Sections
  generate_W:
    Workflow Options:
      --n-samples VAL (as int, required)
                        Generate projection matrix W ~ N(0, 1/sqrt(n)), shape (n x B). Run ONCE before processing any chromosome.  W depends only on n (total sample size) and B — not on any
                        variant data. n_samples is passed directly as a parameter; no VCF reading is needed. All 22 chromosomes reuse the same W so that per-chromosome stochastic genotype
                        samples can be arithmetically merged for meta-analysis.
      --output-dir VAL (as str, required)
      --B 10000 (as int)
      --seed 123 (as int)
  process_block:
    Workflow Options:
      --vcf-base VAL (as str, required)
                        Process one chromosome: VCF → stochastic genotype sample U → PLINK2 pgen.  Memory-safe chunked processing: For each VCF part (one .bgz file = one chunk): G_chunk (n ×
                        p_chunk) → U_chunk = W.T @ G_chunk (B × p_chunk) → scale columns to [0,2] → stream to plink2 via named pipe Peak memory: W (~880 MB) + G_chunk (~8.8 GB) + U_chunk (~4
                        GB) ≈ 14 GB  Block boundary QC (before any computation): QC1 — Continuity: end[i] == start[i+1] for all consecutive blocks QC2 — Coverage:   every VCF variant falls in
                        [start, end) of exactly one block Directory containing VCF (.bgz) files, one or more per chromosome
      --vcf-prefix VAL (as str, required)
                        Shared filename prefix of all VCF files (everything before "chr{N}:" or "chr{N}.")
      --ld-block-file VAL (as str, required)
                        Path to LD block BED file (e.g. EUR_LD_blocks.bed)
      --chrom VAL (as int, required)
      --block-start VAL (as int, required)
      --block-end VAL (as int, required)
      --output-dir VAL (as str, required)
      --W-matrix VAL (as str, required)
      --B 10000 (as int)
      --maf-min 0.0005 (as float)
      --mac-min 5 (as int)
      --msng-min 0.05 (as float)
      --sample-list "" (optional path to sample ID file)
```

# Setup and Global Parameters

In [ ]:
[global]
parameter: cwd        = path("output")
parameter: job_size   = 1
parameter: walltime   = "4h"
parameter: mem        = "32G"
parameter: numThreads = 8
parameter: container  = ""
parameter: plink2_bin  = "plink2"
parameter: walltime    = "24:00:00"
parameter: mem         = "32G"
parameter: numThreads  = 8

import re
from sos.utils import expand_size

entrypoint = (
    'micromamba run -a "" -n' + ' ' +
    re.sub(r'(_apptainer:latest|_docker:latest|\.sif)$', '', container.split('/')[-1])
) if container else ""

cwd = path(f'{cwd:a}')


## `generate_W`

In [ ]:
[generate_W]
# Generate projection matrix W ~ N(0, 1/sqrt(n)), shape (n x B).
# Run ONCE before processing any chromosome.
#
# W depends only on n (total sample size) and B — not on any variant data.
# n_samples is passed directly as a parameter; no VCF reading is needed.
# All 22 chromosomes reuse the same W so that per-chromosome stochastic
# genotype samples can be arithmetically merged for meta-analysis.
parameter: n_samples = int
parameter: output_dir    = str
parameter: B         = 10000
parameter: seed      = 123

import os
input:  []
output: f'{output_dir}/W_B{B}.npy'
task: trunk_workers = 1, trunk_size = 1, walltime = '00:05:00', mem = '4G', cores = 1
python: expand = "${ }", stdout = f'{_output:n}.stdout', stderr = f'{_output:n}.stderr'

    import numpy as np
    import os

    n      = ${n_samples}
    B      = ${B}
    seed   = ${seed}
    W_out  = "${_output}"

    # ── Generate W ~ N(0, 1/sqrt(n)) ─────────────────────────────
    # Convention: W = np.random.normal(0, 1/np.sqrt(n), size=(n, B))
    # W is shared across all chromosomes — do not regenerate per chromosome.
    print(f"Generating W ~ N(0, 1/sqrt({n})),  shape ({n}, {B}),  seed={seed}")
    np.random.seed(seed)
    W = np.random.normal(0, 1.0 / np.sqrt(n), size=(n, B)).astype(np.float32)

    os.makedirs(os.path.dirname(os.path.abspath(W_out)), exist_ok=True)
    np.save(W_out, W)
    print(f"Saved: {W_out}")
    print(f"Shape: {W.shape},  size: {os.path.getsize(W_out)/1e9:.2f} GB")

## `process_block`


In [ ]:
[process_block_1]
# Read BED file and write one .block coord file per LD block.
parameter: vcf_base      = str
parameter: vcf_prefix    = str
parameter: ld_block_file = str
parameter: chrom         = 0
parameter: output_dir    = str
parameter: W_matrix      = str
parameter: B             = 10000
parameter: maf_min       = 0.0005
parameter: mac_min       = 5
parameter: msng_min      = 0.05
parameter: sample_list   = ""

import os

def _read_blocks(bed, chrom_filter):
    blocks = []
    with open(bed) as fh:
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split()
            c = parts[0]
            if not (c.startswith("chr") and c[3:].isdigit()):
                continue
            cnum = int(c[3:])
            if not (1 <= cnum <= 22):
                continue
            if chrom_filter != 0 and cnum != chrom_filter:
                continue
            blocks.append((c, int(parts[1]), int(parts[2])))
    if not blocks:
        raise ValueError(f"No blocks found for chrom={chrom_filter} in {bed}")
    return blocks

_blocks = _read_blocks(ld_block_file, chrom)
print(f"  {len(_blocks)} LD blocks queued")

import tempfile
block_dir = tempfile.mkdtemp(prefix="sos_blocks_")

input:  []
output: [f"{block_dir}/{c}_{s}_{e}.block" for c, s, e in _blocks]
python: expand = "${ }"
    import os, json
    blocks = ${_blocks!r}
    block_dir = "${block_dir}"
    for c, s, e in blocks:
        path = os.path.join(block_dir, f"{c}_{s}_{e}.block")
        with open(path, "w") as fh:
            fh.write(f"{c}\n{s}\n{e}\n")


## `process_block_2`


In [ ]:
[process_block_2]
parameter: vcf_base      = str
parameter: vcf_prefix    = str
parameter: ld_block_file = str
# Cohort identifier prefix for output files (e.g. ADSP.R5.EUR)
parameter: cohort_id     = "ADSP.R5.EUR"
parameter: output_dir    = str
parameter: W_matrix      = str
parameter: B             = 10000
parameter: maf_min       = 0.0005
parameter: mac_min       = 5
parameter: msng_min      = 0.05
parameter: sample_list   = ""

input:  output_from("process_block_1"), group_by = 1
task: trunk_workers = 1, trunk_size = 1, walltime = walltime, mem = mem, cores = numThreads
python: expand = "${ }"

    import numpy as np
    import os
    import subprocess
    from math import nan
    from cyvcf2 import VCF
    from os import listdir

    # Read block coordinates from input file
    with open("${_input}") as fh:
        lines = fh.read().splitlines()
    chrm_str    = lines[0]
    block_start = int(lines[1])
    block_end   = int(lines[2])

    vcf_base    = "${vcf_base}"
    vcf_prefix  = "${vcf_prefix}"
    plink2_bin  = "${plink2_bin}"
    W_path      = "${W_matrix}"
    B           = ${B}
    maf_min     = ${maf_min}
    mac_min     = ${mac_min}
    msng_min    = ${msng_min}
    sample_list = "${sample_list}"

    block_tag  = f"{chrm_str}_{block_start}_{block_end}"
    cohort_id  = "${cohort_id}"
    output_dir = f"${output_dir}/{chrm_str}/{block_tag}"

    os.makedirs(output_dir, exist_ok=True)
    log_path = os.path.join(output_dir, f"{block_tag}.log")
    import sys
    log_fh = open(log_path, "w")
    sys.stdout = log_fh
    sys.stderr = log_fh

    # ── Load sample subset (optional) ─────────────────────────────
    sample_subset = None
    if sample_list:
        if not os.path.exists(sample_list):
            raise FileNotFoundError(f"sample_list not found: {sample_list}")
        with open(sample_list) as fh:
            sample_subset = set(line.strip() for line in fh if line.strip())
        print(f"  Sample subset: {len(sample_subset):,} samples")

    # ── Helpers ───────────────────────────────────────────────────
    def get_vcf_files(chrm_str):
        files = sorted([
            os.path.join(vcf_base, x)
            for x in listdir(vcf_base)
            if x.endswith(".bgz") and (
                x.startswith(vcf_prefix + chrm_str + ":") or
                x.startswith(vcf_prefix + chrm_str + ".")
            )
        ])
        if not files:
            raise FileNotFoundError(f"No VCF files for {chrm_str} in {vcf_base}")
        return files

    def fill_missing_col_means(G):
        col_means = np.nanmean(G, axis=0)
        return np.where(np.isnan(G), col_means, G)

    # ── Step 1: Scan variants in block ────────────────────────────
    print(f"[1/4] Scanning {chrm_str} [{block_start:,}, {block_end:,}) ...")
    vcf_files = get_vcf_files(chrm_str)
    region    = f"{chrm_str}:{block_start+1}-{block_end}"

    var_info  = []
    n_samples = None

    for vf in vcf_files:
        vcf = VCF(vf)
        if sample_subset is not None:
            vcf_samples = vcf.samples
            keep_idx = [i for i, s in enumerate(vcf_samples) if s in sample_subset]
            if not keep_idx:
                raise ValueError(f"No sample_list samples in {os.path.basename(vf)}")
            vcf.set_samples([vcf_samples[i] for i in keep_idx])
        if n_samples is None:
            n_samples = len(vcf.samples)
        for var in vcf(region):
            if not (block_start <= var.POS - 1 < block_end):
                continue
            if len(var.ALT) != 1:
                continue
            dosage = [
                sum(x[0:2])
                for x in [[nan if v == -1 else v for v in gt]
                           for gt in var.genotypes]
            ]
            if np.nanvar(dosage) == 0:
                continue
            counts    = [np.nansum([2 - x for x in dosage]), np.nansum(dosage)]
            nan_count = int(np.sum(np.isnan(dosage)))
            n_non_na  = len(dosage) - nan_count
            mac       = min(counts)
            maf       = mac / n_non_na
            af        = float(np.nansum(dosage)) / (2 * n_non_na)
            msng_rate = nan_count / (n_non_na + nan_count)
            if maf < maf_min or mac < mac_min or msng_rate > msng_min:
                continue
            var_info.append({
                "chr": var.CHROM, "pos": var.POS,
                "ref": var.REF,   "alt": var.ALT[0],
                "af":  round(float(af), 6),
                "id":  f"{var.CHROM}:{var.POS}:{var.REF}:{var.ALT[0]}",
            })
        vcf.close()
    print(f"  {len(var_info):,} variants passing filters  (n={n_samples:,})")

    if not var_info:
        raise ValueError(f"No passing variants in {chrm_str} [{block_start:,}, {block_end:,})")

    # ── Step 2: Load W ────────────────────────────────────────────
    print(f"[2/4] Loading W ...")
    W = np.load(W_path)
    if W.shape != (n_samples, B):
        raise ValueError(f"W shape mismatch: {W.shape} vs ({n_samples},{B})")
    W = W.astype(np.float32)
    print(f"  W: {W.shape}")

    # ── Step 3: Stream → pgen ─────────────────────────────────────
    print(f"[3/4] Streaming → pgen ...")
    prefix        = os.path.join(output_dir, f"{cohort_id}.{block_tag}_stocld_panel_tmp")
    sorted_prefix = os.path.join(output_dir, f"{cohort_id}.{chrm_str}_stocld_panel")
    dosage_path   = prefix + ".dosage"
    psam_path     = prefix + ".psam"
    map_path      = prefix + ".map"
    meta_path     = os.path.join(output_dir, f"{cohort_id}.{block_tag}_stocld_panel.meta")

    with open(meta_path, "w") as fh:
        fh.write(f"source_n_samples={n_samples}\nB={B}\n")
        fh.write(f"chrom={chrm_str}\nblock_start={block_start}\nblock_end={block_end}\n")
    with open(psam_path, "w") as fh:
        fh.write("#FID IID\n")
        for i in range(1, B + 1):
            fh.write(f"S{i} S{i}\n")
    with open(map_path, "w") as fh:
        for v in var_info:
            fh.write(f"{v['chr'].replace('chr','')}\t{v['id']}\t0\t{v['pos']}\n")

    var_lookup = {v["id"]: v for v in var_info}
    var_id_set = set(var_lookup)

    # Write dosage to a real file (plink2 --import-dosage requires two-pass read)
    print(f"  Writing dosage file ...", flush=True)
    total = 0
    with open(dosage_path, "w", buffering=1 << 20) as fout:
        for vf in vcf_files:
            arr, var_keys = [], []
            vcf_obj = VCF(vf)
            for var in vcf_obj(region):
                if not (block_start <= var.POS - 1 < block_end):
                    continue
                if len(var.ALT) != 1:
                    continue
                vid = f"{var.CHROM}:{var.POS}:{var.REF}:{var.ALT[0]}"
                if vid not in var_id_set:
                    continue
                dosage = [
                    sum(x[0:2])
                    for x in [[nan if v == -1 else v for v in gt]
                              for gt in var.genotypes]
                ]
                arr.append(dosage)
                var_keys.append(vid)
            vcf_obj.close()
            if not arr:
                continue
            G_chunk = np.array(arr, dtype=np.float32).T
            G_chunk = fill_missing_col_means(G_chunk)
            U_chunk = W.T @ G_chunk
            del G_chunk
            col_min = U_chunk.min(axis=0)
            col_max = U_chunk.max(axis=0)
            denom   = col_max - col_min
            denom[denom == 0] = 1.0
            U_chunk = 2.0 * (U_chunk - col_min) / denom
            U_chunk = np.round(U_chunk, 4)
            for j, vid in enumerate(var_keys):
                v = var_lookup[vid]
                vals = " ".join(f"{x:.4f}" for x in U_chunk[:, j])
                fout.write(f"{vid} {v['alt']} {v['ref']} {vals}\n")
            total += len(var_keys)
            del U_chunk
            print(f"  {os.path.basename(vf)}: {total:,} variants written", flush=True)
    print(f"  Dosage file written: {total:,} variants", flush=True)

    result = subprocess.run([
        plink2_bin,
        "--import-dosage", dosage_path, "format=1", "noheader",
        "--psam", psam_path, "--map", map_path,
        "--make-pgen", "--out", prefix,
    ])
    if os.path.exists(dosage_path):
        os.remove(dosage_path)
    if result.returncode != 0:
        raise RuntimeError(f"plink2 import failed (exit {result.returncode})")

    result = subprocess.run([
        plink2_bin, "--pfile", prefix,
        "--sort-vars", "--make-pgen", "--out", sorted_prefix,
    ])
    if result.returncode != 0:
        raise RuntimeError(f"plink2 sort failed (exit {result.returncode})")

    for ext in [".pgen", ".pvar", ".psam", ".log", ".map"]:
        fp = prefix + ext
        if os.path.exists(fp):
            os.remove(fp)
    print(f"  Output: {sorted_prefix}.{{pgen,pvar,psam}}")

    # ── Step 4: Allele frequencies ────────────────────────────────
    print(f"[4/4] Writing afreq ...")
    afreq_path = os.path.join(output_dir, f"{cohort_id}.{block_tag}_stocld.afreq")
    obs_ct     = 2 * n_samples
    with open(afreq_path, "w") as fh:
        fh.write("#CHROM\tID\tREF\tALT\tALT_FREQS\tOBS_CT\n")
        for v in var_info:
            fh.write(f"{v['chr'].replace('chr','')}\t{v['id']}\t{v['ref']}\t{v['alt']}\t"
                     f"{v['af']:.6f}\t{obs_ct}\n")
    print(f"  Wrote: {os.path.basename(afreq_path)}")
    print(f"\nDone: {chrm_str} [{block_start:,}, {block_end:,})  →  {sorted_prefix}.{{pgen,pvar,psam}}")
